# Combined TI × LI × SI Pipeline (aligned with final_wingx + wingx-smid-kmeans)

**Merges Time-of-Day Index (TI), Location Index (LI), and Seasonality Index (SI) with raw indices and multipliers**

**Input**: `lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet`  
**Output**: `combined_ti_li_si_output.csv`

**Final schema**: Full audit trail with raw indices AND multipliers per CTS framework

**Scope**: 16 priority corridors (6 Light Jet + 10 Super Midsize Jet) × actual months × 7 DOW × 6 TOD (data-driven, not synthetic)

**Key alignment**: Hours filters (LJ ≥1.5, SMID ≥2.5) matching final_wingx and wingx-smid-kmeans notebooks

## Configuration

In [1]:
import pandas as pd
import numpy as np

# ─── CONFIGURATION REFERENCE ────────────────────────────────────────────────────
# To match different sources, use these presets:
#
# OPTION A: Comprehensive Market (RECOMMENDED — CURRENT)
#   → 16 LJ models + 2024-2025 years
#   → Covers full LJ market (100%), recent data patterns
#   → SI values will differ from wingx-lj-kmeans (different data volume)
#   → Best for: Forward pricing across full LJ segment
#
# OPTION B: Replicate wingx-lj-kmeans exactly
#   → 6 partial strings: ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra']
#   → SELECTED_YEARS = None (all years 2023-2025)
#   → Corridor format: " ➔ " (arrow character, not ->)
#   → SI values match wingx-lj-kmeans exactly (102k missions, Phenom-only)
#   → Best for: Matching wingx-lj-kmeans SI values exactly
#
# OPTION C: final_wingx style (16 models + 2024-2025)
#   → Same as Option A (this is Option A)
#
# Switch: Update LIGHT_JET_MODELS and SELECTED_YEARS below
# ────────────────────────────────────────────────────────────────────────────────

# Input file (local)
RAW_PARQUET = 'gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet'

# Time periods (OPTION A: Comprehensive Market)
SELECTED_YEARS = [2024, 2025]  # Recent 2 years (Option A)
# To match wingx-lj-kmeans: SELECTED_YEARS = None  # All years (Option B)

MEAN_CELL_SHARE = 1 / 42  # 6 TOD × 7 DOW = 42 cells per corridor

# Filtering thresholds
MIN_MISSIONS_LJ = 20      # LJ: min annual missions per corridor
MIN_MISSIONS_SMID = 30    # SMID: min annual missions per corridor
MIN_HOURS_LJ = 1.5        # Hours filter for Light Jet
MIN_HOURS_SMID = 2.5      # Hours filter for SMID

# Aircraft models (OPTION A: Comprehensive Market — 16 exact model names)
LIGHT_JET_MODELS = [
    'Phenom 300', 'Phenom 300-E',
    'Citation CJ3', 'Citation CJ3+',
    'Citation CJ4', 'Hawker 400XP',
    'Citation CJ2+', 'Citation Ultra',
    'Citation V', '400-A',
    'Citation CJ4 Gen2', 'Citation Encore+'
]
# ^ OPTION A: 16 exact models (full market ~323k missions)
# To match wingx-lj-kmeans (Option B), replace with:
# LIGHT_JET_MODELS = ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra']

WU_OPERATORS = [
    'Wheels Up Private Jets', 'Wheels Up LLC',
    'Mountain Aviation', 'Alante Air Charter'
]

# Time categories
DAY_ORDER    = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
BUCKET_ORDER = ['00:00-06:59', '07:00-08:59', '09:00-12:59',
                '13:00-15:59', '16:00-18:59', '19:00-23:59']
MONTH_NAMES  = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

# TI Score mapping (ratio_vs_mean → slab → score)
TI_SCORE_MAP = {
    "> 0 to <= 0.5x":      1,
    "> 0.5x to 0.6x":      2,
    "> 0.6x to 0.75x":     3,
    "> 0.75x to 1.0x":     4,
    "> 1.0x to 1.1x":      5,
    "> 1.1x to 1.25x":     6,
    "> 1.25x to 1.5x":     7,
    "> 1.5x to 1.75x":     8,
    "> 1.75x to 2.0x":     9,
    "> 2.0x to 3.0x":      10,
    "> 3.0x":              10,
}
TI_BINS   = [0, 0.5, 0.6, 0.75, 1.0, 1.1, 1.25, 1.5, 1.75, 2.0, 3.0, float('inf')]
TI_LABELS = list(TI_SCORE_MAP.keys())

# SI Multiplier mapping (seasonality_index → slab → multiplier)
SI_MULTIPLIER_MAP = {
    "< 0.75x":            0.75,
    "0.75x to < 0.90x":   0.90,
    "0.90x to < 1.10x":   1.00,
    "1.10x to < 1.30x":   1.15,
    "1.30x to < 1.60x":   1.30,
    ">= 1.60x":           1.50,
}
SI_BINS   = [0, 0.75, 0.90, 1.10, 1.30, 1.60, float('inf')]
SI_LABELS = list(SI_MULTIPLIER_MAP.keys())

# LI Multiplier mapping (LI_index → slab → multiplier)
LI_MULTIPLIER_MAP = {
    "<= 0.75x":           0.85,
    "> 0.75x to 1.00x":   0.95,
    "> 1.00x to 1.50x":   1.00,
    "> 1.50x to 2.50x":   1.10,
    "> 2.50x to 4.00x":   1.20,
    "> 4.00x":            1.30,
}
LI_BINS   = [0, 0.75, 1.00, 1.50, 2.50, 4.00, float('inf')]
LI_LABELS = list(LI_MULTIPLIER_MAP.keys())

# ─── CTS TOGGLE MAP (CONFIGURABLE - TUPLE FORMAT) ────────────────────────────────
# Easy to edit: (min, max, toggle%) — copy/paste from Excel!
# Format: {segment: [(min, max, toggle%), ...]}
# Add/edit/remove rows easily — just modify the tuples

CTS_TOGGLE_MAP = {
    'Light Jet': [
        (0, 0.25, -15),     (0.25, 0.5, -15),    (0.5, 0.75, -15),    (0.75, 1, -15),
        (1, 1.25, -10),     (1.25, 1.5, -10),    (1.5, 1.75, -5),     (1.75, 2, -5),
        (2, 2.25, 0),       (2.25, 2.5, 0),      (2.5, 2.75, 0),      (2.75, 3, 0),
        (3, 3.25, 0),       (3.25, 3.5, 0),      (3.5, 3.75, 0),      (3.75, 4, 0),
        (4, 4.25, 0),       (4.25, 4.5, 0),      (4.5, 4.75, 0),      (4.75, 5, 5),
        (5, 5.25, 5),       (5.25, 5.5, 5),      (5.5, 5.75, 10),     (5.75, 6, 10),
        (6, 6.25, 10),      (6.25, 6.5, 15),     (6.5, 6.75, 15),     (6.75, 7, 15),
        (7, 7.25, 20),      (7.25, 7.5, 20),     (7.5, 7.75, 20),     (7.75, 8, 25),
        (8, 8.25, 25),      (8.25, 8.5, 25),     (8.5, 8.75, 30),     (8.75, 9, 30),
        (9, 9.25, 30),      (9.25, 9.5, 35),     (9.5, 9.75, 35),     (9.75, 10, 35),
        (10, float('inf'), 35),
    ],
    'Super Midsize Jet': [
        (0, 0.25, -15),     (0.25, 0.5, -15),    (0.5, 0.75, -15),    (0.75, 1, -15),
        (1, 1.25, -10),     (1.25, 1.5, -10),    (1.5, 1.75, -10),    (1.75, 2, -5),
        (2, 2.25, -5),      (2.25, 2.5, -5),     (2.5, 2.75, 0),      (2.75, 3, 0),
        (3, 3.25, 0),       (3.25, 3.5, 0),      (3.5, 3.75, 0),      (3.75, 4, 0),
        (4, 4.25, 5),       (4.25, 4.5, 5),      (4.5, 4.75, 5),      (4.75, 5, 5),
        (5, 5.25, 5),       (5.25, 5.5, 5),      (5.5, 5.75, 5),      (5.75, 6, 5),
        (6, 6.25, 10),      (6.25, 6.5, 10),     (6.5, 6.75, 10),     (6.75, 7, 10),
        (7, 7.25, 10),      (7.25, 7.5, 10),     (7.5, 7.75, 10),     (7.75, 8, 10),
        (8, 8.25, 10),      (8.25, 8.5, 10),     (8.5, 8.75, 10),     (8.75, 9, 10),
        (9, 9.25, 10),      (9.25, 9.5, 15),     (9.5, 9.75, 15),     (9.75, 10, 15),
        (10, float('inf'), 15),
    ]
}


# Priority corridors
LJ_CORRIDORS = [
    'Denver->LA Basin',           'LA Basin->Denver',
    'Charlotte->South Florida',   'South Florida->Charlotte',
    'Chicago->South Florida',     'South Florida->Chicago',
]

SMID_CORRIDORS = [
    'New York City->South Florida', 'South Florida->New York City',
    'Denver->South Florida',        'South Florida->Denver',
    'Chicago->South Florida',       'South Florida->Chicago',
    'Phoenix Valley->Boston',       'Boston->Phoenix Valley',
    'LA Basin->New York City',      'New York City->LA Basin',
]

print("✓ Configuration loaded (OPTION A: Comprehensive Market)")
print(f"  LJ models: {len(LIGHT_JET_MODELS)} (full market coverage)")
print(f"  Year range: {SELECTED_YEARS}")
print(f"  LJ hours: >= {MIN_HOURS_LJ} | min_missions > {MIN_MISSIONS_LJ}")
print(f"  SMID hours: >= {MIN_HOURS_SMID} | min_missions > {MIN_MISSIONS_SMID}")
print(f"  CTS Toggle map loaded: {len(CTS_TOGGLE_MAP)} segments")
print(f"\n📍 To edit CTS toggle mapping (easy!):")
print(f"  Edit CTS_TOGGLE_MAP in Cell 2")
print(f"  Change tuples: (min, max, toggle%)")
print(f"  Example: (5, 5.25, 0) → (5, 5.25, 5)  [increase toggle from 0% to 5%]")
print(f"\n  Light Jet ranges: {len(CTS_TOGGLE_MAP['Light Jet'])} tuples")
print(f"  Super Midsize Jet ranges: {len(CTS_TOGGLE_MAP['Super Midsize Jet'])} tuples")

✓ Configuration loaded (OPTION A: Comprehensive Market)
  LJ models: 12 (full market coverage)
  Year range: [2024, 2025]
  LJ hours: >= 1.5 | min_missions > 20
  SMID hours: >= 2.5 | min_missions > 30
  CTS Toggle map loaded: 2 segments

📍 To edit CTS toggle mapping (easy!):
  Edit CTS_TOGGLE_MAP in Cell 2
  Change tuples: (min, max, toggle%)
  Example: (5, 5.25, 0) → (5, 5.25, 5)  [increase toggle from 0% to 5%]

  Light Jet ranges: 41 tuples
  Super Midsize Jet ranges: 41 tuples


## Step 1: Load & Preprocess

In [2]:
df = pd.read_parquet(RAW_PARQUET)
# df['FlightDate_local'] = pd.to_datetime(df['FlightDate_local'])
# df['year_local']  = df['FlightDate_local'].dt.year
# df['month_local'] = df['FlightDate_local'].dt.month

print(f"✓ Loaded {len(df):,} records")
print(f"  Date range: {df['FlightDate_local'].min().date()} to {df['FlightDate_local'].max().date()}")

# Filter to inter-metro (EXACT MATCH: City_Mapping style — NO OTHER_METRO exclusion)
data = df[
    df['year_local'].isin(SELECTED_YEARS) &
    (df['FromMetro'] != df['ToMetro'])
].copy()
print(f"✓ Filtered to {SELECTED_YEARS}")

# Build corridors (format: "Metro->Metro" at metro level)
data['Route']   = data['FromMetro'] + '->' + data['ToMetro']
data['corridor_city_mapping'] = data['FromCity'] + '->' + data['ToCity']  # City-level mapping
data['is_wu']   = data['Operator'].isin(WU_OPERATORS)

df['Route']   = df['FromMetro'] + '->' + df['ToMetro']
df['corridor_city_mapping'] = data['FromCity'] + '->' + df['ToCity']

# Categorize days and time buckets
data['day_name']    = pd.Categorical(data['dow_local'],         categories=DAY_ORDER,    ordered=True)
data['hour_bucket'] = pd.Categorical(data['hour_bucket_local'], categories=BUCKET_ORDER, ordered=True)

print(f"✓ Preprocessed {len(data):,} inter-metro records")

✓ Loaded 2,140,706 records
  Date range: 2023-12-31 to 2025-12-31
✓ Filtered to [2024, 2025]
✓ Preprocessed 1,865,186 inter-metro records


In [3]:
df.columns

Index(['FlightDate_utc', 'Operator', 'FromAirport', 'FromState', 'ToAirport',
       'ToState', 'Hours', 'aircraft_tailsign', 'aircraft_model',
       'aircraft_segment', 'fuel_uplift_usg', 'FlightDate_ET', 'year', 'month',
       'dow', 'hour', 'FromMetro', 'ToMetro', 'latitude', 'longitude',
       'FlightDate_local', 'hour_local', 'dow_local', 'hour_bucket_local',
       'month_local', 'year_local', 'FromCity', 'ToCity', 'DayOfYear', 'Route',
       'corridor_city_mapping'],
      dtype='object')

## Step 2: Segment Split

In [4]:
df[['FromAirport', 'ToAirport', 'FromMetro', 'ToMetro', 'FromState', 'ToState', 
    'FromCity', 'ToCity', 'DayOfYear', 'Route', 
    'corridor_city_mapping']]

,FromAirport,ToAirport,FromMetro,ToMetro,FromState,ToState,FromCity,ToCity,DayOfYear,Route,corridor_city_mapping
0,KTEX,KBJC,Denver,Denver,CO,CO,TELLURIDE,Denver,Sunday,Denver->Denver,NaN
1,KJAC,KSDL,Denver,Phoenix Valley,WY,AZ,Jackson,SCOTTSDALE,Saturday,Denver->Phoenix Valley,Jackson->SCOTTSDALE
2,KORL,KCHO,North Florida,DMV,FL,VA,ORLANDO,CHARLOTTESVILLE,Friday,North Florida->DMV,ORLANDO->CHARLOTTESVILLE
3,KFCM,KSDL,Minneapolis,Phoenix Valley,MN,AZ,Minneapolis,SCOTTSDALE,Wednesday,Minneapolis->Phoenix Valley,Minneapolis->SCOTTSDALE
4,KSNA,KSLC,LA Basin,Denver,CA,UT,SANTA ANA,SALT LAKE CITY,Friday,LA Basin->Denver,SANTA ANA->SALT LAKE CITY
...,...,...,...,...,...,...,...,...,...,...,...
2140701,KPGD,KTYQ,South Florida,Chicago,FL,IN,PUNTA GORDA,Indianapolis,Tuesday,South Florida->Chicago,PUNTA GORDA->Indianapolis
2140702,KGPC,KMSP,OTHER_METRO,Minneapolis,IN,MN,Greencastle,Minneapolis,Tuesday,OTHER_METRO->Minneapolis,Greencastle->Minneapolis
2140703,KGPC,KYIP,OTHER_METRO,Detroit,IN,MI,Greencastle,DETROIT,Friday,OTHER_METRO->Detroit,Greencastle->DETROIT
2140704,KFAR,KGPC,Minneapolis,OTHER_METRO,ND,IN,Fargo,Greencastle,Friday,Minneapolis->OTHER_METRO,Fargo->Greencastle


In [5]:
df['icao_corridor']   = df['FromAirport'] + '->' + df['ToAirport']

In [6]:
df[df["Route"]=="Denver->LA Basin"]['icao_corridor'].nunique()

1423

In [7]:
df[df["Operator"].isin(WU_OPERATORS)]['icao_corridor'].nunique()

36611

In [8]:
df[(df["Operator"].isin(WU_OPERATORS)) & (df["Route"].isin(LJ_CORRIDORS + SMID_CORRIDORS))]['icao_corridor'].nunique()

1878

In [9]:
df.shape

(2140706, 32)

In [10]:
data_t = df[
    df['year_local'].isin(SELECTED_YEARS) &
    (df['FromMetro'] != df['ToMetro'])
    # (data['Operator'].isin(WU_OPERATORS))
].copy()

In [11]:
data_t[data_t["Route"]=="Denver->LA Basin"]['corridor_city_mapping'].nunique()

1142

In [12]:
data[df['year_local'].isin(SELECTED_YEARS)]

/var/tmp/ipykernel_6835/2736047709.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data[df['year_local'].isin(SELECTED_YEARS)]


,FlightDate_utc,Operator,FromAirport,FromState,ToAirport,ToState,Hours,aircraft_tailsign,aircraft_model,aircraft_segment,...,month_local,year_local,FromCity,ToCity,DayOfYear,Route,corridor_city_mapping,is_wu,day_name,hour_bucket
1,2024-03-16T22:58:00.000Z,Mountain Aviation,KJAC,WY,KSDL,AZ,1.400000,N903UP,Citation X,Super Midsize Jet,...,3,2024,Jackson,SCOTTSDALE,Saturday,Denver->Phoenix Valley,Jackson->SCOTTSDALE,True,Saturday,16:00-18:59
2,2024-03-01T22:32:00.000Z,Mountain Aviation,KORL,FL,KCHO,VA,1.516667,N907TX,Citation X,Super Midsize Jet,...,3,2024,ORLANDO,CHARLOTTESVILLE,Friday,North Florida->DMV,ORLANDO->CHARLOTTESVILLE,True,Friday,16:00-18:59
3,2024-03-20T16:09:00.000Z,Mountain Aviation,KFCM,MN,KSDL,AZ,2.533333,N907TX,Citation X,Super Midsize Jet,...,3,2024,Minneapolis,SCOTTSDALE,Wednesday,Minneapolis->Phoenix Valley,Minneapolis->SCOTTSDALE,True,Wednesday,09:00-12:59
4,2024-03-22T18:12:00.000Z,Mountain Aviation,KSNA,CA,KSLC,UT,1.300000,N908UP,Citation X,Super Midsize Jet,...,3,2024,SANTA ANA,SALT LAKE CITY,Friday,LA Basin->Denver,SANTA ANA->SALT LAKE CITY,True,Friday,09:00-12:59
5,2024-03-20T22:18:00.000Z,Mountain Aviation,KIAD,VA,KPNE,PA,0.516667,N909UP,Citation X,Super Midsize Jet,...,3,2024,Washington,Philadelphia,Wednesday,DMV->Philadelphia,Washington->Philadelphia,True,Wednesday,16:00-18:59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2140701,2025-11-11T18:19:00.000Z,Integrated Flight Resources Inc,KPGD,FL,KTYQ,IN,2.266667,N293CK,Phenom 300-E,Light Jet,...,11,2025,PUNTA GORDA,Indianapolis,Tuesday,South Florida->Chicago,PUNTA GORDA->Indianapolis,False,Tuesday,13:00-15:59
2140702,2025-10-21T20:22:00.000Z,NetJets,KGPC,IN,KMSP,MN,1.333333,N872QS,Citation Longitude,Super Midsize Jet,...,10,2025,Greencastle,Minneapolis,Tuesday,OTHER_METRO->Minneapolis,Greencastle->Minneapolis,False,Tuesday,16:00-18:59
2140703,2025-10-03T16:28:00.000Z,Best Jets International,KGPC,IN,KYIP,MI,0.733333,N896BB,Challenger 300,Super Midsize Jet,...,10,2025,Greencastle,DETROIT,Friday,OTHER_METRO->Detroit,Greencastle->DETROIT,False,Friday,09:00-12:59
2140704,2025-10-17T17:23:00.000Z,Northwest Aviation LLC,KFAR,ND,KGPC,IN,1.616667,N415SD,Citation CJ4,Light Jet,...,10,2025,Fargo,Greencastle,Friday,Minneapolis->OTHER_METRO,Fargo->Greencastle,False,Friday,09:00-12:59


In [13]:
df["Operator"].unique()

array(['Mountain Aviation', 'Waste Connections US Inc',
       'Williams Communities LLC', ..., '525B-Operator', 'TWY17-Operator',
       'XBCAM-Operator'], shape=(6710,), dtype=object)

In [14]:
smid_data = data[data['aircraft_segment'] == 'Super Midsize Jet'].copy()
lj_data   = data[data['aircraft_model'].isin(LIGHT_JET_MODELS)].copy()

print(f"✓ LJ records (all): {len(lj_data):,}")
print(f"  LJ models in data: {lj_data['aircraft_model'].nunique()}")
print(f"✓ SMID records (all): {len(smid_data):,}")

✓ LJ records (all): 621,687
  LJ models in data: 12
✓ SMID records (all): 940,570


# ─── MODEL AUDIT: Find ALL distinct LJ models in dataset (like final_wingx Cell 1.5) ────


In [15]:

# 1. Get ALL Light Jet records (before any model filtering)
df_lj_all = data[data['aircraft_segment'] == 'Light Jet'].copy()

# 2. Get distinct models and their mission counts
lj_all_models = df_lj_all['aircraft_model'].value_counts().reset_index()
lj_all_models.columns = ['aircraft_model', 'mission_count']

print(f"\n📋 DISTINCT LIGHT JET MODELS IN DATASET ({len(lj_all_models)} found):")
print("─" * 70)
display(lj_all_models)

# 3. Show which models from your config are in the dataset (EXACT MATCH)
print(f"\n🔍 CONFIGURATION MODEL MATCHING (EXACT):")
print(f"Config has: {LIGHT_JET_MODELS}")
print(f"\nExact matches in dataset:")
for model in LIGHT_JET_MODELS:
    matches = lj_all_models[lj_all_models['aircraft_model'] == model]
    if not matches.empty:
        count = matches['mission_count'].values[0]
        pct = (count / len(df_lj_all)) * 100
        print(f"  ✓ '{model}': {count:,} missions ({pct:.1f}%)")
    else:
        print(f"  ✗ '{model}': NOT FOUND in dataset")

# 4. Pattern matching (like final_wingx) - show what REGEX patterns would match
print(f"\n🔍 PATTERN MATCHING (SUBSTRING/REGEX - like final_wingx):")
pattern_list = ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra', '400']
print(f"Testing patterns: {pattern_list}\n")

for pattern in pattern_list:
    matches = lj_all_models[lj_all_models['aircraft_model'].str.contains(pattern, case=False, na=False)]
    if not matches.empty:
        model_list = matches['aircraft_model'].tolist()
        total_missions = matches['mission_count'].sum()
        print(f"  '{pattern}' matches:")
        for model in model_list:
            count = matches[matches['aircraft_model'] == model]['mission_count'].values[0]
            print(f"    • {model}: {count:,} missions")
    else:
        print(f"  '{pattern}': NO MATCHES")

# 5. Summary stats
config_coverage = df_lj_all[df_lj_all['aircraft_model'].isin(LIGHT_JET_MODELS)]
print(f"\n📊 COVERAGE SUMMARY:")
print(f"  Total LJ missions (all models): {len(df_lj_all):,}")
print(f"  LJ missions matching config: {len(config_coverage):,}")
print(f"  Coverage: {(len(config_coverage) / len(df_lj_all) * 100):.1f}%")
print(f"  Unique models in config: {len([m for m in LIGHT_JET_MODELS if m in lj_all_models['aircraft_model'].values])}")
print(f"  Unique models NOT in config: {len(lj_all_models) - len([m for m in LIGHT_JET_MODELS if m in lj_all_models['aircraft_model'].values])}")


📋 DISTINCT LIGHT JET MODELS IN DATASET (46 found):
──────────────────────────────────────────────────────────────────────


,aircraft_model,mission_count
0,Phenom 300,231796
1,Citation CJ3,80873
2,no model available,74381
3,Phenom 300-E,55802
4,Citation CJ3+,49719
5,Citation CJ4,45691
6,Citation Ultra,32047
7,400-A,32046
8,Citation V,30909
9,400-XP,27777



🔍 CONFIGURATION MODEL MATCHING (EXACT):
Config has: ['Phenom 300', 'Phenom 300-E', 'Citation CJ3', 'Citation CJ3+', 'Citation CJ4', 'Hawker 400XP', 'Citation CJ2+', 'Citation Ultra', 'Citation V', '400-A', 'Citation CJ4 Gen2', 'Citation Encore+']

Exact matches in dataset:
  ✓ 'Phenom 300': 231,796 missions (25.1%)
  ✓ 'Phenom 300-E': 55,802 missions (6.0%)
  ✓ 'Citation CJ3': 80,873 missions (8.7%)
  ✓ 'Citation CJ3+': 49,719 missions (5.4%)
  ✓ 'Citation CJ4': 45,691 missions (4.9%)
  ✓ 'Hawker 400XP': 22,782 missions (2.5%)
  ✓ 'Citation CJ2+': 17,348 missions (1.9%)
  ✓ 'Citation Ultra': 32,047 missions (3.5%)
  ✓ 'Citation V': 30,909 missions (3.3%)
  ✓ '400-A': 32,046 missions (3.5%)
  ✓ 'Citation CJ4 Gen2': 14,004 missions (1.5%)
  ✓ 'Citation Encore+': 8,670 missions (0.9%)

🔍 PATTERN MATCHING (SUBSTRING/REGEX - like final_wingx):
Testing patterns: ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra', '400']

  'Phenom 300' matches:
    • Phenom 300: 231,7

In [16]:
def compute_si(df, corridors, cabin_label, min_hours, min_missions):
    """
    Compute Seasonality Index multiplier per corridor × month.

    Follows OPTION A: Comprehensive Market
    - 16 LJ models + 2024-2025 years (full LJ market coverage)
    - SI values differ from wingx-lj-kmeans due to larger data volume

    Steps:
    1. Filter by min_hours (1.5 for LJ, 2.5 for SMID)
    2. Group by corridor × month
    3. Filter to corridors with > min_missions annual missions
    4. Compute monthly_density = monthly_flights / annual_flights
    5. Compute SI = monthly_density / (1/12)
    6. Bin to SI_multiplier

    Returns: corridor, cabin, month, month_num, monthly_density, seasonality_index, si_slab, SI_multiplier
    """
    # Filter by hours AND corridor list
    df_corr = df[(df['Route'].isin(corridors)) & (df['Hours'] >= min_hours)].copy()

    # Step 1: Monthly flight counts
    monthly = df_corr.groupby(['Route','month_local']).size().unstack(fill_value=0)
    monthly = monthly.reindex(columns=range(1,13), fill_value=0)

    # Step 2: FILTER - min_missions threshold
    annual = monthly.sum(axis=1)
    mask_min_missions = annual > min_missions
    monthly_filtered = monthly[mask_min_missions].copy()
    annual_filtered = annual[mask_min_missions].copy()

    if len(monthly_filtered) == 0:
        print(f"  ⚠️  [{cabin_label}] No corridors passed filters (hours >= {min_hours} AND missions > {min_missions})")
        return pd.DataFrame()

    print(f"  ✓ [{cabin_label}] {len(monthly_filtered)} corridors (OPTION A: {len(LIGHT_JET_MODELS)} models, {SELECTED_YEARS})")

    # Step 3: Normalize to monthly density
    density = monthly_filtered.div(annual_filtered, axis=0)

    # Step 4: Create seasonality index
    si = density / (1/12)

    # Convert to long format
    si_long = si.stack().rename('seasonality_index').reset_index()
    si_long.columns = ['corridor','month_num','seasonality_index']

    # Add monthly_density
    density_long = density.stack().rename('monthly_density').reset_index()
    density_long.columns = ['corridor','month_num','monthly_density']
    si_long = si_long.merge(density_long, on=['corridor','month_num'])

    # Bin into SI slab label
    si_long['si_slab'] = pd.cut(
        si_long['seasonality_index'],
        bins=SI_BINS,
        labels=SI_LABELS,
        right=False
    )

    # Map slab to multiplier
    si_long['SI_multiplier'] = si_long['si_slab'].map(SI_MULTIPLIER_MAP).astype(float)

    # Add month name
    si_long['month'] = si_long['month_num'].map(MONTH_NAMES)
    si_long['cabin'] = cabin_label

    return si_long[['corridor','cabin','month','month_num','monthly_density','seasonality_index','si_slab','SI_multiplier']]

print("\n🔄 Computing SI (OPTION A: Comprehensive Market):")
print(f"  LJ: {len(LIGHT_JET_MODELS)} models, {SELECTED_YEARS}, min_missions > {MIN_MISSIONS_LJ}")
print(f"  SMID: All SMID models, {SELECTED_YEARS}, min_missions > {MIN_MISSIONS_SMID}")
print(f"\n📍 To replicate wingx-lj-kmeans (Option B):")
print(f"  Change LIGHT_JET_MODELS to 6 partial strings and SELECTED_YEARS = None")
lj_si   = compute_si(lj_data,   LJ_CORRIDORS,   'Light Jet',        MIN_HOURS_LJ,   MIN_MISSIONS_LJ)
smid_si = compute_si(smid_data, SMID_CORRIDORS, 'Super Midsize Jet', MIN_HOURS_SMID, MIN_MISSIONS_SMID)

print(f"\n✓ LJ SI: {len(lj_si)} rows")
print(f"✓ SMID SI: {len(smid_si)} rows")


🔄 Computing SI (OPTION A: Comprehensive Market):
  LJ: 12 models, [2024, 2025], min_missions > 20
  SMID: All SMID models, [2024, 2025], min_missions > 30

📍 To replicate wingx-lj-kmeans (Option B):
  Change LIGHT_JET_MODELS to 6 partial strings and SELECTED_YEARS = None
  ✓ [Light Jet] 6 corridors (OPTION A: 12 models, [2024, 2025])
  ✓ [Super Midsize Jet] 10 corridors (OPTION A: 12 models, [2024, 2025])

✓ LJ SI: 72 rows
✓ SMID SI: 120 rows


## Step 4: TI Computation (with hours filter)

In [17]:
def compute_ti(df, corridors, cabin_label):
    """
    Compute TI Score (0-10) per corridor × day × hour_bucket.
    Follows City_Mapping style: NO hours filter (all flight durations included).
    Returns: corridor, cabin, DOW, TOD, cell_share, ratio_vs_mean, ti_slab, TI_score
    """
    df_corr = df[df['Route'].isin(corridors)].copy()

    # Count flights in each (Route, day, time) cell
    grouped = (
        df_corr.groupby(['Route','day_name','hour_bucket'], observed=False)
        .size()
        .reset_index(name='flight_count')
    )

    # Total flights per corridor (across all cells)
    grouped['total_corridor_flights'] = grouped.groupby('Route')['flight_count'].transform('sum')

    # Cell share (what fraction of corridor flights are in this cell)
    grouped['cell_share'] = grouped['flight_count'] / grouped['total_corridor_flights']

    # Ratio vs mean (how much above/below flat 1/42)
    grouped['ratio_vs_mean'] = grouped['cell_share'] / MEAN_CELL_SHARE

    # Bin into TI slab label (right=True for exclusive left, inclusive right)
    grouped['ti_slab'] = pd.cut(
        grouped['ratio_vs_mean'],
        bins=TI_BINS,
        labels=TI_LABELS,
        right=True,
        ordered=False
    )

    # Map slab label to TI score using the map
    grouped['TI_score'] = grouped['ti_slab'].map(TI_SCORE_MAP).astype('Int64')

    # Mark zero-flight cells as 0 (these should not occur with observed=False, but safety check)
    grouped.loc[grouped['flight_count'] == 0, ['ti_slab', 'TI_score']] = [None, 0]

    # Rename for final output
    grouped = grouped.rename(columns={'Route':'corridor','day_name':'DOW','hour_bucket':'TOD'})
    grouped['cabin'] = cabin_label

    return grouped[['corridor','cabin','DOW','TOD','cell_share','ratio_vs_mean','ti_slab','TI_score']]

lj_ti   = compute_ti(lj_data,   LJ_CORRIDORS,   'Light Jet')
smid_ti = compute_ti(smid_data, SMID_CORRIDORS, 'Super Midsize Jet')

print(f"✓ LJ TI: {len(lj_ti)} rows")
print(f"✓ SMID TI: {len(smid_ti)} rows")

# Validation
print(f"\n📋 Sample LJ TI scores (top by intensity):")
if 'Denver->LA Basin' in lj_ti['corridor'].values:
    sample = lj_ti[lj_ti['corridor'] == 'Denver->LA Basin'].nlargest(5, 'ratio_vs_mean')[['DOW','TOD','ratio_vs_mean','ti_slab','TI_score']]
    display(sample)
else:
    print(f"  Denver->LA Basin not in corridors")

✓ LJ TI: 252 rows
✓ SMID TI: 420 rows

📋 Sample LJ TI scores (top by intensity):


,DOW,TOD,ratio_vs_mean,ti_slab,TI_score
122,Sunday,09:00-12:59,2.572358,> 2.0x to 3.0x,10
110,Friday,09:00-12:59,2.549593,> 2.0x to 3.0x,10
86,Monday,09:00-12:59,2.276423,> 2.0x to 3.0x,10
104,Thursday,09:00-12:59,2.173984,> 2.0x to 3.0x,10
116,Saturday,09:00-12:59,2.145528,> 2.0x to 3.0x,10


## Step 5: LI Computation (with hours filter)

In [18]:
def compute_li(df, corridors, cabin_label):
    """
    Compute Location Index multiplier per corridor.
    LI_index = (corridor WU share) / (network WU share)
    Follows City_Mapping style: NO hours filter (all flight durations included).
    Returns: corridor, cabin, corridor_wu_share, network_wu_share, LI_index, li_slab, LI_multiplier
    """
    # Network-wide WU operator share (no hours filter)
    network_wu_share = df['is_wu'].sum() / len(df)

    # Filter to priority corridors
    df_corr = df[df['Route'].isin(corridors)]

    # Corridor-level aggregation
    agg = (
        df_corr.groupby('Route')
        .agg(
            corridor_total=('is_wu', 'count'),
            corridor_wu=('is_wu', 'sum')
        )
        .reset_index()
    )

    # WU share on corridor
    agg['corridor_wu_share'] = agg['corridor_wu'] / agg['corridor_total']

    # LI Index
    agg['LI_index'] = agg['corridor_wu_share'] / network_wu_share

    # Bin into LI slab label
    agg['li_slab'] = pd.cut(
        agg['LI_index'],
        bins=LI_BINS,
        labels=LI_LABELS,
        right=True,
        include_lowest=True,
        ordered=False
    )

    # Map slab to LI multiplier
    agg['LI_multiplier'] = agg['li_slab'].map(LI_MULTIPLIER_MAP).astype(float)

    agg = agg.rename(columns={'Route':'corridor'})
    agg['cabin'] = cabin_label
    agg['network_wu_share'] = network_wu_share

    print(f"[{cabin_label}] Network WU share (all flights): {network_wu_share:.4%}")

    return agg[['corridor','cabin','corridor_wu_share','network_wu_share','LI_index','li_slab','LI_multiplier']]

lj_li   = compute_li(lj_data,   LJ_CORRIDORS,   'Light Jet')
smid_li = compute_li(smid_data, SMID_CORRIDORS, 'Super Midsize Jet')

print(f"✓ LJ LI: {len(lj_li)} rows")
print(f"✓ SMID LI: {len(smid_li)} rows")

[Light Jet] Network WU share (all flights): 5.0921%
[Super Midsize Jet] Network WU share (all flights): 2.3077%
✓ LJ LI: 6 rows
✓ SMID LI: 10 rows


## Step 6: Combine All Data

In [19]:
# Concatenate by index type
si_all = pd.concat([lj_si, smid_si], ignore_index=True)
ti_all = pd.concat([lj_ti, smid_ti], ignore_index=True)
li_all = pd.concat([lj_li, smid_li], ignore_index=True)

print(f"✓ SI combined: {len(si_all)} rows")
print(f"✓ TI combined: {len(ti_all)} rows")
print(f"✓ LI combined: {len(li_all)} rows")

✓ SI combined: 192 rows
✓ TI combined: 672 rows
✓ LI combined: 16 rows


## Step 7: Build Final Table

In [20]:
# ─── ACTUAL MONTHS (data-driven, not synthetic) ───────────────────────────────────────
# Extract actual months present in the flight data (like City_Mapping uses actual dow_local)
actual_months_df = data[['month_local']].drop_duplicates().rename(columns={'month_local': 'month_num'})
actual_months_df['month'] = actual_months_df['month_num'].map(MONTH_NAMES)
actual_months_df = actual_months_df.sort_values('month_num').reset_index(drop=True)

print(f"✓ Actual months in data: {sorted(actual_months_df['month_num'].unique())}")
print(f"  Month names: {', '.join(actual_months_df['month'].unique())}")

# Start with TI (corridor × day × time) and cross with actual months
final = ti_all.merge(actual_months_df, how='cross')

print(f"✓ After cross-join with actual months: {len(final):,} rows")

# Join SI (by corridor × cabin × month) — only for corridors that passed MIN_MISSIONS filter
final = final.merge(
    si_all[['corridor','cabin','month','monthly_density','seasonality_index','si_slab','SI_multiplier']],
    on=['corridor','cabin','month'],
    how='left'
)

print(f"✓ After SI join: {len(final):,} rows")
print(f"  SI nulls (corridors filtered out by MIN_MISSIONS): {final['SI_multiplier'].isnull().sum():,}")

# Join LI (by corridor × cabin)
final = final.merge(
    li_all[['corridor','cabin','corridor_wu_share','network_wu_share','LI_index','li_slab','LI_multiplier']],
    on=['corridor','cabin'],
    how='left'
)

print(f"✓ After LI join: {len(final):,} rows")

# Join city mapping (by corridor)
corridor_city_map = data[['Route', 'corridor_city_mapping']].drop_duplicates().rename(columns={'Route': 'corridor'})
final = final.merge(corridor_city_map, on='corridor', how='left')

print(f"✓ After city mapping join: {len(final):,} rows")

✓ Actual months in data: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
  Month names: Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec
✓ After cross-join with actual months: 8,064 rows
✓ After SI join: 8,064 rows
  SI nulls (corridors filtered out by MIN_MISSIONS): 0
✓ After LI join: 8,064 rows
✓ After city mapping join: 4,023,936 rows


## Step 8: Finalize Output

In [21]:
# Rename cabin → segment
final = final.rename(columns={'cabin':'segment', 'month':'Month'})

# Calculate CTS Score (Combined Tuning Score)
# CTS = TI_score × SI_multiplier × LI_multiplier
final['CTS_score'] = final['TI_score'] * final['SI_multiplier'] * final['LI_multiplier']

# Map CTS_score to toggle percentage by segment (tuple format)
# def map_cts_to_toggle(row):
#     """Map CTS_score to toggle % based on segment using CTS_TOGGLE_MAP tuples"""
#     segment = row['segment']
#     cts_score = row['CTS_score']

#     # If CTS_score is null, toggle is null
#     if pd.isna(cts_score) or segment not in CTS_TOGGLE_MAP:
#         return np.nan

#     # Iterate through tuples: (min, max, toggle%)
#     for min_val, max_val, toggle_pct in CTS_TOGGLE_MAP[segment]:
#         if min_val <= cts_score < max_val:
#             return toggle_pct

#     return np.nan

# # Apply the mapping
# final['CTS_toggle_%'] = final.apply(map_cts_to_toggle, axis=1)

# Apply the mapping (vectorized with np.select for speed)
conditions = []
choices = []

for segment in final['segment'].unique():
    if segment not in CTS_TOGGLE_MAP:
        continue
    
    segment_mask = final['segment'] == segment
    for min_val, max_val, toggle_pct in CTS_TOGGLE_MAP[segment]:
        condition = segment_mask & (final['CTS_score'] >= min_val) & (final['CTS_score'] < max_val)
        conditions.append(condition)
        choices.append(toggle_pct)

final['CTS_toggle_%'] = np.select(conditions, choices, default=np.nan)


# Select and reorder final columns (with corridor_city_mapping)
# final = final[[
#     'corridor', 'corridor_city_mapping', 'segment', 'TOD', 'DOW', 'Month',
#     'cell_share', 'ratio_vs_mean', 'ti_slab', 'TI_score',
#     'monthly_density', 'seasonality_index', 'si_slab', 'SI_multiplier',
#     'corridor_wu_share', 'network_wu_share', 'LI_index', 'li_slab', 'LI_multiplier',
#     'CTS_score', 'CTS_toggle_%'
# ]]

final = final[[
    'corridor', 'corridor_city_mapping', 'segment', 'TOD', 'DOW', 'Month',
    'TI_score','seasonality_index', 'SI_multiplier', 'LI_index', 'LI_multiplier',
    'CTS_score', 'CTS_toggle_%'
]]

# Sort for readability
final = final.sort_values(
    ['segment', 'corridor', 'Month', 'DOW', 'TOD']
).reset_index(drop=True)

# Validation
print(f"\n✅ FINAL TABLE COMPLETE (with corridor_city_mapping, CTS_score and CTS_toggle_%)")
print(f"\n📊 Summary:")
print(f"  Rows: {len(final):,}")
print(f"  Columns: {len(final.columns)}")
print(f"  Unique corridors: {final['corridor'].nunique()}")
print(f"  Unique segments: {final['segment'].nunique()}")
print(f"  Unique months: {final['Month'].nunique()} (actual from data: {sorted(final['Month'].unique())})")
print(f"  Expected grid (actual): {final['corridor'].nunique()} corridors × {final['Month'].nunique()} months × 7 DOW × 6 TOD = {final['corridor'].nunique() * final['Month'].nunique() * 7 * 6:,} rows")

print(f"\n🔍 Hours filter impact (SI):")
si_pass_rate = (final['SI_multiplier'].notna().sum() / len(final)) * 100
print(f"  SI_multiplier not null: {final['SI_multiplier'].notna().sum():,} rows ({si_pass_rate:.1f}%)")
print(f"  SI_multiplier nulls (filtered by MIN_MISSIONS): {final['SI_multiplier'].isnull().sum():,} rows")

print(f"\n🔍 Other null checks:")
# print(f"  cell_share nulls: {final['cell_share'].isnull().sum()}")
# print(f"  ti_slab nulls: {final['ti_slab'].isnull().sum()}")
print(f"  LI_multiplier nulls: {final['LI_multiplier'].isnull().sum()}")
print(f"  corridor_city_mapping nulls: {final['corridor_city_mapping'].isnull().sum()}")
print(f"  CTS_score nulls (where SI is null): {final['CTS_score'].isnull().sum()}")
print(f"  CTS_toggle_% nulls: {final['CTS_toggle_%'].isnull().sum()}")

print(f"\n📋 Segment breakdown:")
for segment in final['segment'].unique():
    count = len(final[final['segment'] == segment])
    si_null = final[(final['segment']==segment) & (final['SI_multiplier'].isnull())].shape[0]
    toggle_null = final[(final['segment']==segment) & (final['CTS_toggle_%'].isnull())].shape[0]
    print(f"  {segment}: {count:,} rows (SI nulls: {si_null:,}, toggle nulls: {toggle_null:,})")

print(f"\n🔢 Index & Score distribution:")
print(f"  TI_score: {final['TI_score'].min():.0f} to {final['TI_score'].max():.0f}")
print(f"  SI_multiplier: {final['SI_multiplier'].min():.2f} to {final['SI_multiplier'].max():.2f}")
print(f"  LI_multiplier: {final['LI_multiplier'].min():.2f} to {final['LI_multiplier'].max():.2f}")
print(f"  CTS_score: {final['CTS_score'].min():.2f} to {final['CTS_score'].max():.2f}")

print(f"\n📊 CTS Score & Toggle Summary (only non-null):")
cts_non_null = final[final['CTS_score'].notna()]
toggle_non_null = final[final['CTS_toggle_%'].notna()]
print(f"  CTS_score rows: {len(cts_non_null):,}")
print(f"    Mean: {cts_non_null['CTS_score'].mean():.2f}")
print(f"    Median: {cts_non_null['CTS_score'].median():.2f}")
print(f"  CTS_toggle_% rows: {len(toggle_non_null):,}")
print(f"    Range: {toggle_non_null['CTS_toggle_%'].min():.0f}% to {toggle_non_null['CTS_toggle_%'].max():.0f}%")
print(f"    Mean: {toggle_non_null['CTS_toggle_%'].mean():.1f}%")
print(f"    Median: {toggle_non_null['CTS_toggle_%'].median():.1f}%")

print(f"\n📍 Toggle % distribution:")
toggle_dist = final[final['CTS_toggle_%'].notna()]['CTS_toggle_%'].value_counts().sort_index()
for toggle_pct, count in toggle_dist.items():
    pct_of_total = (count / len(toggle_non_null)) * 100
    print(f"  {toggle_pct:>3.0f}%: {count:,} rows ({pct_of_total:>5.1f}%)")

print(f"\n📊 Sample rows (with city mapping and CTS_score):")
display(final[final['CTS_toggle_%'].notna()][['corridor', 'corridor_city_mapping', 'segment', 'TOD', 'DOW', 'Month', 'TI_score', 'SI_multiplier', 'LI_multiplier', 'CTS_score', 'CTS_toggle_%']].head(25))


✅ FINAL TABLE COMPLETE (with corridor_city_mapping, CTS_score and CTS_toggle_%)

📊 Summary:
  Rows: 4,023,936
  Columns: 13
  Unique corridors: 14
  Unique segments: 2
  Unique months: 12 (actual from data: ['Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep'])
  Expected grid (actual): 14 corridors × 12 months × 7 DOW × 6 TOD = 7,056 rows

🔍 Hours filter impact (SI):
  SI_multiplier not null: 4,023,936 rows (100.0%)
  SI_multiplier nulls (filtered by MIN_MISSIONS): 0 rows

🔍 Other null checks:
  LI_multiplier nulls: 0
  corridor_city_mapping nulls: 0
  CTS_score nulls (where SI is null): 0
  CTS_toggle_% nulls: 0

📋 Segment breakdown:
  Light Jet: 2,467,080 rows (SI nulls: 0, toggle nulls: 0)
  Super Midsize Jet: 1,556,856 rows (SI nulls: 0, toggle nulls: 0)

🔢 Index & Score distribution:
  TI_score: 0 to 10
  SI_multiplier: 0.75 to 1.50
  LI_multiplier: 0.85 to 1.20
  CTS_score: 0.00 to 18.00

📊 CTS Score & Toggle Summary (only non-null):
  CTS_score r

,corridor,corridor_city_mapping,segment,TOD,DOW,Month,TI_score,SI_multiplier,LI_multiplier,CTS_score,CTS_toggle_%
0,Charlotte->South Florida,Greensboro->WEST PALM BEACH,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
1,Charlotte->South Florida,Charleston->NAPLES,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
2,Charlotte->South Florida,Myrtle Beach->Sarasota Bradenton,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
3,Charlotte->South Florida,CONCORD->VERO BEACH,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
4,Charlotte->South Florida,Greensboro->FORT LAUDERDALE,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
5,Charlotte->South Florida,Charleston->FORT LAUDERDALE,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
6,Charlotte->South Florida,Roxboro->Marco Island,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
7,Charlotte->South Florida,Charleston->Sarasota Bradenton,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
8,Charlotte->South Florida,Columbia->STUART,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
9,Charlotte->South Florida,WINSTON SALEM->Marco Island,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0


## Step 9: Export

In [22]:
output_path = "combined_ti_li_si_output.csv"
final.to_csv(output_path, index=False)

print(f"\n✅ EXPORTED: {output_path}")
print(f"\n📋 Verification:")
check = pd.read_csv(output_path)
print(f"  Rows: {len(check):,}")
print(f"  Columns: {len(check.columns)}")
print(f"  SI data rows: {check['SI_multiplier'].notna().sum():,}")
print(f"  CTS_score rows: {check['CTS_score'].notna().sum():,}")
print(f"  CTS_toggle_% rows: {check['CTS_toggle_%'].notna().sum():,}")

print(f"\n✓ Pipeline complete (OPTION A: Comprehensive Market)")
print(f"  Config: {len(LIGHT_JET_MODELS)} LJ models + {SELECTED_YEARS}")
print(f"  LJ hours filter: >= {MIN_HOURS_LJ}")
print(f"  SMID hours filter: >= {MIN_HOURS_SMID}")
print(f"  Month selection: ACTUAL months from data (data-driven, not synthetic)")

print(f"\n🎯 CTS Score (Combined Tuning Score):")
print(f"  Formula: TI_score × SI_multiplier × LI_multiplier")
print(f"  Range: {check['CTS_score'].min():.2f} to {check['CTS_score'].max():.2f}")
print(f"  Mean: {check[check['CTS_score'].notna()]['CTS_score'].mean():.2f}")

print(f"\n💰 CTS Toggle % (Price Adjustment):")
print(f"  Formula: Map CTS_score to toggle % by segment")
print(f"  Range: {check['CTS_toggle_%'].min():.0f}% to {check['CTS_toggle_%'].max():.0f}%")
toggle_vals = check[check['CTS_toggle_%'].notna()]['CTS_toggle_%']
print(f"  Mean: {toggle_vals.mean():.1f}%")
print(f"  Unique values: {toggle_vals.nunique()} different toggle percentages")

print(f"\n📍 Configuration Reference:")
print(f"  OPTION A (current): 16 models + 2024-2025 years + ACTUAL months")
print(f"    → Full market coverage, recent data, data-driven month selection")
print(f"    → SI differs from wingx-lj-kmeans (different data volume)")
print(f"  OPTION B: 6 partial strings + all years (2023-2025) + ACTUAL months")
print(f"    → Matches wingx-lj-kmeans exactly")
print(f"    → Narrow (Phenom-300 only, ~102k missions)")
print(f"\n  To switch: Edit Cell 2 configuration")
print(f"\n⚙️  To customize CTS toggle mapping:")
print(f"  Edit CTS_TOGGLE_MAP in Cell 2 (segment names and toggle % values)")


✅ EXPORTED: combined_ti_li_si_output.csv

📋 Verification:
  Rows: 4,023,936
  Columns: 13
  SI data rows: 4,023,936
  CTS_score rows: 4,023,936
  CTS_toggle_% rows: 4,023,936

✓ Pipeline complete (OPTION A: Comprehensive Market)
  Config: 12 LJ models + [2024, 2025]
  LJ hours filter: >= 1.5
  SMID hours filter: >= 2.5
  Month selection: ACTUAL months from data (data-driven, not synthetic)

🎯 CTS Score (Combined Tuning Score):
  Formula: TI_score × SI_multiplier × LI_multiplier
  Range: 0.00 to 18.00
  Mean: 4.44

💰 CTS Toggle % (Price Adjustment):
  Formula: Map CTS_score to toggle % by segment
  Range: -15% to 35%
  Mean: 2.5%
  Unique values: 11 different toggle percentages

📍 Configuration Reference:
  OPTION A (current): 16 models + 2024-2025 years + ACTUAL months
    → Full market coverage, recent data, data-driven month selection
    → SI differs from wingx-lj-kmeans (different data volume)
  OPTION B: 6 partial strings + all years (2023-2025) + ACTUAL months
    → Matches wing

# month extraction

In [23]:
def get_month_dataset(final_df, months=None):
    """
    Extract dataset for specific month(s).

    Parameters:
    -----------
    final_df : DataFrame
        The final combined TI/LI/SI dataset
    months : int, str, list, or None
        - None: returns full dataset
        - int (1-12): returns single month by number
        - str ('Jan', 'Feb'...): returns single month by name
        - list: returns multiple months [1, 2, 'Mar'] or ['Jan', 'Feb', 'Mar']

    Returns:
    --------
    DataFrame with filtered data
    """
    if months is None:
        return final_df.copy()

    # Standardize months to list
    if isinstance(months, (int, str)):
        months = [months]

    # Month mapping
    month_name_to_num = {v: k for k, v in MONTH_NAMES.items()}
    num_to_name = MONTH_NAMES

    # Convert all inputs to month names (for filtering)
    month_names = []
    for m in months:
        if isinstance(m, int) and 1 <= m <= 12:
            month_names.append(num_to_name[m])
        elif isinstance(m, str) and m in month_name_to_num:
            month_names.append(m)
        else:
            print(f"  ⚠️  Invalid month: {m}. Use 1-12 or Jan-Dec")

    # Filter by Month column (which contains names like 'Jan', 'Feb', etc.)
    result = final_df[final_df['Month'].isin(month_names)].copy()

    print(f"✓ Extracted {len(result):,} rows for months: {month_names}")
    print(f"  Corridors: {result['corridor'].nunique()}")
    print(f"  Segments: {result['segment'].nunique()}")
    print(f"  Rows per corridor: {len(result) / result['corridor'].nunique():.0f}")

    return result.reset_index(drop=True)


In [24]:
# Single month
may_df = get_month_dataset(final, months=[5,6])
may_df.to_csv('month_may_june.csv', index=False)

# # Season (Jun-Aug)
# summer_df = get_month_dataset(final, months=[6, 7, 8])
# summer_df.to_csv('season_summer.csv', index=False)

# By name
# mar_df = get_month_dataset(final, months='Mar')


✓ Extracted 670,656 rows for months: ['May', 'Jun']
  Corridors: 14
  Segments: 2
  Rows per corridor: 47904


In [25]:
# Read the CSV
corridors_list = pd.read_csv("corridors_list.csv")

# Convert the specific column from your CSV to lower case
# Replace 'column_name' with the actual name of the column in corridors_list.csv
search_values = corridors_list.iloc[:, 0].str.lower()

# Filter the map
mask = corridor_city_map["corridor_city_mapping"].str.lower().isin(search_values)
result = corridor_city_map[mask]
result

,corridor,corridor_city_mapping
2,North Florida->DMV,ORLANDO->CHARLOTTESVILLE
3,Minneapolis->Phoenix Valley,Minneapolis->SCOTTSDALE
4,LA Basin->Denver,SANTA ANA->SALT LAKE CITY
5,DMV->Philadelphia,Washington->Philadelphia
6,Denver->Pittsburgh,Jackson->Pittsburgh
...,...,...
2136404,Atlanta->DMV,ALBANY->RICHMOND
2136946,North Florida->Nashville,THOMASVILLE->Lexington
2138657,Charlotte->Nashville,FLORENCE->Columbus
2138726,Atlanta->Nashville,Knoxville->Cleveland


In [26]:
may_df[may_df["corridor"]=="South Florida->Chicago"]

,corridor,corridor_city_mapping,segment,TOD,DOW,Month,TI_score,seasonality_index,SI_multiplier,LI_index,LI_multiplier,CTS_score,CTS_toggle_%
363216,South Florida->Chicago,Marco Island->Holland,Light Jet,00:00-06:59,Monday,Jun,1,0.634761,0.75,0.560916,0.85,0.6375,-15.0
363217,South Florida->Chicago,NAPLES->VALPARAISO,Light Jet,00:00-06:59,Monday,Jun,1,0.634761,0.75,0.560916,0.85,0.6375,-15.0
363218,South Florida->Chicago,NAPLES->South Bend,Light Jet,00:00-06:59,Monday,Jun,1,0.634761,0.75,0.560916,0.85,0.6375,-15.0
363219,South Florida->Chicago,NAPLES->Chicago,Light Jet,00:00-06:59,Monday,Jun,1,0.634761,0.75,0.560916,0.85,0.6375,-15.0
363220,South Florida->Chicago,FORT MYERS->Chicago,Light Jet,00:00-06:59,Monday,Jun,1,0.634761,0.75,0.560916,0.85,0.6375,-15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
611263,South Florida->Chicago,Melbourne->GARY,Super Midsize Jet,19:00-23:59,Sunday,May,1,1.182336,1.15,0.982077,0.95,1.0925,-10.0
611264,South Florida->Chicago,FORT LAUDERDALE->Freeport,Super Midsize Jet,19:00-23:59,Sunday,May,1,1.182336,1.15,0.982077,0.95,1.0925,-10.0
611265,South Florida->Chicago,BOCA RATON->ROCKFORD,Super Midsize Jet,19:00-23:59,Sunday,May,1,1.182336,1.15,0.982077,0.95,1.0925,-10.0
611266,South Florida->Chicago,STUART->Lafayette,Super Midsize Jet,19:00-23:59,Sunday,May,1,1.182336,1.15,0.982077,0.95,1.0925,-10.0


In [27]:
# Diagnostic: Charlotte->South Florida for May only
may_data = get_month_dataset(final, months='May')
char_sf = may_data[may_data['corridor'] == 'Charlotte->South Florida']

print(f"Charlotte->South Florida in May:")
print(f"  Total rows: {len(char_sf):,}")
print(f"  Unique segments: {char_sf['segment'].nunique()}")
print(f"  Unique TODs: {char_sf['TOD'].nunique()}")
print(f"  Unique DOWs: {char_sf['DOW'].nunique()}")
print(f"  Expected: 1 segment × 6 TOD × 7 DOW = {1 * 6 * 7}")

print(f"\nUnique TOD values:")
print(f"  {char_sf['TOD'].unique()}")

print(f"\nUnique DOW values:")
print(f"  {char_sf['DOW'].unique()}")

# Check for duplicates
duplicates = char_sf.groupby(['segment', 'TOD', 'DOW']).size()
if (duplicates > 1).any():
    print(f"\n⚠️  DUPLICATES FOUND:")
    print(duplicates[duplicates > 1])


✓ Extracted 335,328 rows for months: ['May']
  Corridors: 14
  Segments: 2
  Rows per corridor: 23952
Charlotte->South Florida in May:
  Total rows: 33,180
  Unique segments: 1
  Unique TODs: 6
  Unique DOWs: 7
  Expected: 1 segment × 6 TOD × 7 DOW = 42

Unique TOD values:
  ['00:00-06:59', '07:00-08:59', '09:00-12:59', '13:00-15:59', '16:00-18:59', '19:00-23:59']
Categories (6, object): ['00:00-06:59' < '07:00-08:59' < '09:00-12:59' < '13:00-15:59' < '16:00-18:59' < '19:00-23:59']

Unique DOW values:
  ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
Categories (7, object): ['Monday' < 'Tuesday' < 'Wednesday' < 'Thursday' < 'Friday' < 'Saturday' < 'Sunday']

⚠️  DUPLICATES FOUND:
segment    TOD          DOW      
Light Jet  00:00-06:59  Monday       790
                        Tuesday      790
                        Wednesday    790
                        Thursday     790
                        Friday       790
                        Saturday     790


/var/tmp/ipykernel_6835/3274096052.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  duplicates = char_sf.groupby(['segment', 'TOD', 'DOW']).size()


In [28]:
may_df[may_df["corridor"]=="Denver->LA Basin"]["corridor_city_mapping"].unique()

array(['EAGLE->San Diego', 'RIFLE->PALM SPRINGS', 'TAOS->SANTA BARBARA',
       ..., 'Colby->Las Vegas', 'Laramie->ONTARIO', 'Burley->San Diego'],
      shape=(1142,), dtype=object)

In [29]:
# ─── METRO-ONLY VERSION (without city mapping) ───────────────────────────────────────
print("\n" + "="*80)
print("METRO-LEVEL ONLY DATASET (no city mapping expansion)")
print("="*80)

# Recreate final table WITHOUT city mapping join
final_metro = ti_all.merge(actual_months_df, how='cross')
final_metro = final_metro.merge(
    si_all[['corridor','cabin','month','monthly_density','seasonality_index','si_slab','SI_multiplier']],
    on=['corridor','cabin','month'],
    how='left'
)
final_metro = final_metro.merge(
    li_all[['corridor','cabin','corridor_wu_share','network_wu_share','LI_index','li_slab','LI_multiplier']],
    on=['corridor','cabin'],
    how='left'
)

# Rename and select columns
final_metro = final_metro.rename(columns={'cabin':'segment', 'month':'Month'})
final_metro['CTS_score'] = final_metro['TI_score'] * final_metro['SI_multiplier'] * final_metro['LI_multiplier']

# Map CTS_score to toggle %
# final_metro['CTS_toggle_%'] = final_metro.apply(map_cts_to_toggle, axis=1)

# Map CTS_score to toggle % (vectorized)
conditions = []
choices = []

for segment in final_metro['segment'].unique():
    if segment not in CTS_TOGGLE_MAP:
        continue
    
    segment_mask = final_metro['segment'] == segment
    for min_val, max_val, toggle_pct in CTS_TOGGLE_MAP[segment]:
        condition = segment_mask & (final_metro['CTS_score'] >= min_val) & (final_metro['CTS_score'] < max_val)
        conditions.append(condition)
        choices.append(toggle_pct)

final_metro['CTS_toggle_%'] = np.select(conditions, choices, default=np.nan)


# Select metro-level only columns
final_metro = final_metro[[
    'corridor', 'segment', 'TOD', 'DOW', 'Month',
    'TI_score', 'SI_multiplier', 'LI_multiplier',
    'CTS_score', 'CTS_toggle_%'
]]

final_metro = final_metro.sort_values(
    ['segment', 'corridor', 'Month', 'DOW', 'TOD']
).reset_index(drop=True)

# Summary
print(f"\n✅ METRO-LEVEL DATASET CREATED")
print(f"\n📊 Comparison:")
print(f"  {'':30} {'WITH City Mapping':>20} {'Metro-Only':>20}")
print(f"  {'-'*30} {'-'*20} {'-'*20}")
print(f"  {'Rows':30} {len(final):>20,} {len(final_metro):>20,}")
print(f"  {'Columns':30} {len(final.columns):>20} {len(final_metro.columns):>20}")
print(f"  {'Unique corridors':30} {final['corridor'].nunique():>20} {final_metro['corridor'].nunique():>20}")
print(f"  {'Segments':30} {final['segment'].nunique():>20} {final_metro['segment'].nunique():>20}")
print(f"  {'File size (approx)':30} {'~1.5 GB':>20} {'~300 KB':>20}")

print(f"\n📋 Metro-Only Columns:")
print(f"  {final_metro.columns.tolist()}")

print(f"\n📍 Sample (first 10 rows):")
display(final_metro.head(10))

# Export metro-only version
metro_output = "combined_ti_li_si_metro_only.csv"
final_metro.to_csv(metro_output, index=False)
print(f"\n✅ EXPORTED: {metro_output}")
print(f"  Rows: {len(final_metro):,}")
print(f"  Size: Metro-level only (no city expansion)")

print(f"\n🔄 Summary of outputs:")
print(f"  1. combined_ti_li_si_output.csv → Full (4M rows with city mapping)")
print(f"  2. {metro_output} → Metro-only (8K rows, no city mapping)")


METRO-LEVEL ONLY DATASET (no city mapping expansion)

✅ METRO-LEVEL DATASET CREATED

📊 Comparison:
                                    WITH City Mapping           Metro-Only
  ------------------------------ -------------------- --------------------
  Rows                                      4,023,936                8,064
  Columns                                          13                   10
  Unique corridors                                 14                   14
  Segments                                          2                    2
  File size (approx)                          ~1.5 GB              ~300 KB

📋 Metro-Only Columns:
  ['corridor', 'segment', 'TOD', 'DOW', 'Month', 'TI_score', 'SI_multiplier', 'LI_multiplier', 'CTS_score', 'CTS_toggle_%']

📍 Sample (first 10 rows):


,corridor,segment,TOD,DOW,Month,TI_score,SI_multiplier,LI_multiplier,CTS_score,CTS_toggle_%
0,Charlotte->South Florida,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
1,Charlotte->South Florida,Light Jet,07:00-08:59,Monday,Apr,3,1.0,1.1,3.3,0.0
2,Charlotte->South Florida,Light Jet,09:00-12:59,Monday,Apr,10,1.0,1.1,11.0,35.0
3,Charlotte->South Florida,Light Jet,13:00-15:59,Monday,Apr,8,1.0,1.1,8.8,30.0
4,Charlotte->South Florida,Light Jet,16:00-18:59,Monday,Apr,4,1.0,1.1,4.4,0.0
5,Charlotte->South Florida,Light Jet,19:00-23:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
6,Charlotte->South Florida,Light Jet,00:00-06:59,Tuesday,Apr,1,1.0,1.1,1.1,-10.0
7,Charlotte->South Florida,Light Jet,07:00-08:59,Tuesday,Apr,2,1.0,1.1,2.2,0.0
8,Charlotte->South Florida,Light Jet,09:00-12:59,Tuesday,Apr,9,1.0,1.1,9.9,35.0
9,Charlotte->South Florida,Light Jet,13:00-15:59,Tuesday,Apr,7,1.0,1.1,7.7,20.0



✅ EXPORTED: combined_ti_li_si_metro_only.csv
  Rows: 8,064
  Size: Metro-level only (no city expansion)

🔄 Summary of outputs:
  1. combined_ti_li_si_output.csv → Full (4M rows with city mapping)
  2. combined_ti_li_si_metro_only.csv → Metro-only (8K rows, no city mapping)


# main data

In [ ]:
"""
Build target pricing dataset from combined_ti_li_si_pipeline_final_1.ipynb output.

Specifications:
- Date range: 2026-05-15 to 2026-06-04
- Pricing category: "Instant Book"
- Timeslots: Named labels + ranges (MORNING, MIDAFTERNOON, etc.)
- Description: "Agnt Works Phase 1 test"
- Routes: Airport level (from raw data)
- Route types: "Airport"
- User type/subtype: "ALL"
- Hourly rates/capacity: BLANK
- SelectionTab: "PERDATE"

FIX: Loads 'final' & 'data' from Jupyter memory (not CSV/parquet files)
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ─── TIMESLOT MAPPING ─────────────────────────────────────────────────────────────
TIMESLOT_MAP = {
    '00:00-06:59': {'label': 'REDEYE-MORNING', 'range': '0000 - 0659'},
    '07:00-08:59': {'label': 'MORNING', 'range': '0700 - 0859'},
    '09:00-12:59': {'label': 'MIDAFTERNOON', 'range': '0900 - 1259'},
    '13:00-15:59': {'label': 'AFTERNOON', 'range': '1300 - 1559'},
    '16:00-18:59': {'label': 'EVENING', 'range': '1600 - 1859'},
    '19:00-23:59': {'label': 'REDEYE-EVENING', 'range': '1900 - 1159'},
}

# ─── DATE RANGE ───────────────────────────────────────────────────────────────────
DATE_START = datetime(2026, 5, 15)
DATE_END = datetime(2026, 6, 4)

# Generate all dates in range
all_dates = []
current_date = DATE_START
while current_date <= DATE_END:
    all_dates.append(current_date)
    current_date += timedelta(days=1)

print(f"[OK] Date range: {DATE_START.strftime('%Y-%m-%d')} to {DATE_END.strftime('%Y-%m-%d')} ({len(all_dates)} days)")

# ─── CHECK: Ensure 'final' and 'data' are in memory from notebook ──────────────────
print("\n[CHECK] Verifying DataFrames from notebook...")
if 'final' not in dir():
    print("[ERROR] 'final' DataFrame not found!")
    print("[INFO] Run the notebook first: combined_ti_li_si_pipeline_final_1.ipynb")
    raise ValueError("Missing 'final' DataFrame from notebook")

if 'data' not in dir():
    print("[ERROR] 'data' DataFrame not found!")
    print("[INFO] Run the notebook first: combined_ti_li_si_pipeline_final_1.ipynb")
    raise ValueError("Missing 'data' DataFrame from notebook")

print(f"[OK] final: {len(final):,} rows")
print(f"[OK] data: {len(data):,} rows")

# ─── BUILD PRICING_DF ──────────────────────────────────────────────────────────────
print("\n[BUILD] Processing...")
pricing_df = final.copy()

# The 'final' DataFrame from notebook already has airport + state columns
# Just check if they exist and use them directly
print("\n[MAP] Checking for airport columns in final DataFrame...")

# List available columns
print(f"  Available columns: {pricing_df.columns.tolist()}")

# Check what airport columns we have
has_from_airport = 'FromAirport' in pricing_df.columns
has_to_airport = 'ToAirport' in pricing_df.columns
has_from_state = 'FromState' in pricing_df.columns
has_to_state = 'ToState' in pricing_df.columns

if not (has_from_airport and has_to_airport and has_from_state and has_to_state):
    print("  [WARN] Missing airport/state columns in final")
    print(f"    FromAirport: {has_from_airport}")
    print(f"    ToAirport: {has_to_airport}")
    print(f"    FromState: {has_from_state}")
    print(f"    ToState: {has_to_state}")
    # Create placeholder columns if missing
    if not has_from_airport:
        pricing_df['FromAirport'] = None
    if not has_to_airport:
        pricing_df['ToAirport'] = None
    if not has_from_state:
        pricing_df['FromState'] = None
    if not has_to_state:
        pricing_df['ToState'] = None
else:
    print("  [OK] All airport columns present")

print(f"  [OK] Mapped {pricing_df['FromAirport'].notna().sum():,} routes to airports")

# ─── BUILD 20 TARGET COLUMNS ──────────────────────────────────────────────────────
print("\n[BUILD] Building 20 target columns...")

# 1-4: Routes (airport) + Types
pricing_df['fromRoute'] = pricing_df['FromAirport']
pricing_df['fromRouteType'] = 'Airport'
pricing_df['toRoute'] = pricing_df['ToAirport']
pricing_df['toRouteType'] = 'Airport'

# 6-7: User type/subtype
pricing_df['usertype'] = 'ALL'
pricing_df['usersubtype'] = 'ALL'

# 8: Pricing category
pricing_df['pricing_category'] = 'Instant Book'

# 9-11: Hourly rates (BLANK)
pricing_df['hourly_rate'] = np.nan
pricing_df['min_hourly_rate'] = np.nan
pricing_df['max_hourly_rate'] = np.nan

# 12: Pricing capacity (BLANK)
pricing_df['pricing_capacity'] = np.nan

# 13-16: Dates & SelectionTab
pricing_df['start_date'] = DATE_START.strftime('%Y-%m-%d')
pricing_df['end_date'] = DATE_END.strftime('%Y-%m-%d')
pricing_df['SelectionTab'] = 'PERDATE'

# selectedDates: comma-separated list of all dates
selected_dates_str = ','.join([d.strftime('%Y-%m-%d') for d in all_dates])
pricing_df['selectedDates'] = selected_dates_str

# 17: todVariance - CONVERT CTS_toggle_% to decimal (0.1 format)
pricing_df['todVariance'] = pricing_df['CTS_toggle_%'] / 100.0

# 18: Timeslots (label + range)
pricing_df['timeslot_label'] = pricing_df['TOD'].map(lambda x: TIMESLOT_MAP.get(x, {}).get('label'))
pricing_df['timeslot_range'] = pricing_df['TOD'].map(lambda x: TIMESLOT_MAP.get(x, {}).get('range'))
pricing_df['timeslots'] = pricing_df['timeslot_label'] + ' (' + pricing_df['timeslot_range'] + ')'

# 19: Description
pricing_df['Description'] = 'Agnt Works Phase 1 test'

# 20: To Route State
pricing_df['to_Route_State'] = pricing_df['ToState']

# ─── SELECT & ORDER 20 COLUMNS ────────────────────────────────────────────────────
print("\n[SELECT] Selecting 20 target columns...")

pricing_output = pricing_df[[
    'fromRoute',              # 1
    'fromRouteType',          # 2
    'toRoute',                # 3
    'toRouteType',            # 4
    'segment',                # 5
    'usertype',               # 6
    'usersubtype',            # 7
    'pricing_category',       # 8
    'hourly_rate',            # 9
    'min_hourly_rate',        # 10
    'max_hourly_rate',        # 11
    'pricing_capacity',       # 12
    'start_date',             # 13
    'end_date',               # 14
    'SelectionTab',           # 15
    'selectedDates',          # 16
    'todVariance',            # 17 - NOW IN DECIMAL FORMAT
    'timeslots',              # 18
    'Description',            # 19
    'to_Route_State',         # 20
]].copy()

# Rename to match target schema
column_names = [
    'fromRoute', 'fromRouteType', 'toRoute', 'toRouteType',
    'aircraft', 'usertype', 'usersubtype', 'pricing_category',
    'hourly_rate', 'min_hourly_rate', 'max_hourly_rate', 'pricing_capacity',
    'start_date', 'end_date', 'SelectionTab', 'selectedDates',
    'todVariance', 'timeslots', 'Description', 'to_Route_State'
]

pricing_output.columns = column_names

# ─── EXPORT ────────────────────────────────────────────────────────────────────────
output_path = "target_pricing_dataset.csv"
pricing_output.to_csv(output_path, index=False)

print(f"\n[EXPORT] {output_path}")
print(f"\n[SUMMARY]")
print(f"  Rows: {len(pricing_output):,}")
print(f"  Columns: {len(pricing_output.columns)}")
print(f"  File size: {pricing_output.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

print(f"\n[SPECS_CONFIRMED]")
print(f"  fromRoute/toRoute: Airport codes")
print(f"  fromRouteType/toRouteType: 'Airport'")
print(f"  aircraft: {pricing_output['aircraft'].unique().tolist()}")
print(f"  usertype/usersubtype: 'ALL'")
print(f"  pricing_category: 'Instant Book'")
print(f"  hourly_rate/min/max/capacity: BLANK (NaN)")
print(f"  start_date: {pricing_output['start_date'].iloc[0]}")
print(f"  end_date: {pricing_output['end_date'].iloc[0]}")
print(f"  SelectionTab: 'PERDATE'")
print(f"  selectedDates: {len(all_dates)} dates ({all_dates[0].strftime('%Y-%m-%d')} to {all_dates[-1].strftime('%Y-%m-%d')})")
print(f"  todVariance: DECIMAL FORMAT (0.1) range {pricing_output['todVariance'].min():.3f} to {pricing_output['todVariance'].max():.3f}")
print(f"  timeslots: {pricing_output['timeslots'].nunique()} unique")
print(f"  Description: 'Agnt Works Phase 1 test'")
print(f"  to_Route_State: State codes")

print(f"\n[SAMPLE]")
print(pricing_output[[
    'fromRoute', 'toRoute', 'aircraft', 'pricing_category',
    'start_date', 'SelectionTab', 'todVariance', 'timeslots'
]].head(10).to_string())

print(f"\n[TIMESLOTS]")
print(pricing_output['timeslots'].value_counts().to_string())

print(f"\n[NULLS]")
null_cols = {}
for col in pricing_output.columns:
    null_count = pricing_output[col].isnull().sum()
    if null_count > 0:
        null_cols[col] = null_count
        pct = (null_count / len(pricing_output)) * 100
        print(f"  {col}: {null_count:,} nulls ({pct:.1f}%)")

if not null_cols:
    print("  No null values [OK]")

print(f"\n[SUCCESS] Dataset complete: {output_path}")


[OK] Date range: 2026-05-15 to 2026-06-04 (21 days)

[CHECK] Verifying DataFrames from notebook...
[OK] final: 4,023,936 rows
[OK] data: 1,865,186 rows

[BUILD] Processing...

[MAP] Checking for airport columns in final DataFrame...
  Available columns: ['corridor', 'corridor_city_mapping', 'segment', 'TOD', 'DOW', 'Month', 'TI_score', 'seasonality_index', 'SI_multiplier', 'LI_index', 'LI_multiplier', 'CTS_score', 'CTS_toggle_%']
  [WARN] Missing airport/state columns in final
    FromAirport: False
    ToAirport: False
    FromState: False
    ToState: False
  [OK] Mapped 0 routes to airports

[BUILD] Building 20 target columns...
